In [1]:
!pip install langchain langchain-community chromadb pypdf sentence-transformers torch transformers accelerate rank_bm25

import os
from typing import List, Dict, Optional
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import re
import uuid
from rank_bm25 import BM25Okapi


INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 76.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━

2025-11-15 14:09:27.756830: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763215767.935589      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763215767.983708      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 9.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 4.1.0
    Uninstalling sentence-transformers-4.1.0:
      Successfully uninstalled sentence-transformers-4.1.0


In [3]:
import os
import re
from typing import List, Optional
import torch
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.schema import Document
from rank_bm25 import BM25Okapi


class LegalSearchAgent:
    # =======================================================================
    #                         INITIALIZATION
    # =======================================================================
    def __init__(self, pdf_folder: str, embeddings: HuggingFaceEmbeddings, db_path: str = "chroma_db", test_mode: bool = True):
        self.pdf_folder = pdf_folder
        self.db_path = db_path
        self.test_mode = test_mode
        self.embeddings = embeddings

        self.vectorstore: Optional[Chroma] = None
        self.bm25: Optional[BM25Okapi] = None
        self.bm25_docs: List[Document] = []
        self.bm25_corpus: List[List[str]] = []

    # =======================================================================
    #                         BUILD VECTOR DATABASE
    # =======================================================================
    def build_vectordb(self):
        all_docs: List[Document] = []

        pdf_files = [f for f in os.listdir(self.pdf_folder) if f.lower().endswith(".pdf") and len(f) > 5]
        if self.test_mode:
            pdf_files = pdf_files[:5]
            print(f"⚠️ TEST MODE: Using only first {len(pdf_files)} PDFs")
        else:
            print(f"Found {len(pdf_files)} PDFs to process.")

        # -------------------------------
        # LOAD PDFs AND ATTACH METADATA
        # -------------------------------
        for idx, pdf in enumerate(pdf_files, 1):
            try:
                print(f"📄 Loading [{idx}/{len(pdf_files)}]: {pdf}")
                loader = PyPDFLoader(os.path.join(self.pdf_folder, pdf))
                pages = loader.load()

                base = pdf.replace(".pdf", "")
                parts = re.split(r"[_\-]", base)
                case_type = "_".join(parts[:-1]) if len(parts) >= 2 else base
                case_year = parts[-1] if parts[-1].isdigit() else "unknown"

                for page in pages:
                    page.metadata["source_file"] = pdf
                    page.metadata["case_number"] = base
                    page.metadata["case_year"] = case_year
                    page.metadata["case_type"] = case_type

                all_docs.extend(pages)

            except Exception as e:
                print(f"❌ Error loading {pdf}: {e}")

        print(f"Loaded {len(all_docs)} pages. Splitting...")

        # -------------------------------
        # SPLIT TEXT AND INJECT METADATA
        # -------------------------------
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=["\n\n", "\n", ".", " ", ""]
        )

        chunks: List[Document] = []
        for page in all_docs:
            split_chunks = splitter.split_documents([page])
            for ch in split_chunks:
                # Copy metadata from original page
                ch.metadata = page.metadata.copy()
                case_type = page.metadata.get("case_type", "")
                case_year = page.metadata.get("case_year", "")
                case_number = page.metadata.get("case_number", "")

                # Inject metadata into text for semantic search
                ch.page_content = f"[Case Type: {case_type}] [Year: {case_year}] [Case Number: {case_number}]\n\n{ch.page_content}"
                chunks.append(ch)

        print(f"✂️ Split into {len(chunks)} chunks.")
        print("🔮 Generating embeddings and building vector DB...")

        # -------------------------------
        # BUILD CHROMA VECTORSTORE
        # -------------------------------
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.db_path
        )
        print(f"✅ Chroma DB saved at {self.db_path}")

        # -------------------------------
        # BUILD BM25 INDEX
        # -------------------------------
        self.bm25_docs = chunks
        self.bm25_corpus = [ch.page_content.split() for ch in chunks]
        self.bm25 = BM25Okapi(self.bm25_corpus)
        print("📌 BM25 index ready.\n")

    # =======================================================================
    #                         CASE NUMBER DETECTION
    # =======================================================================
    def _detect_case_number(self, query: str) -> Optional[str]:
        pattern = r"(CPLA|C\.A\.|Cr\.A|C\.P\.|HCA|RFA)[\s\-]*\d+[\s/]*(?:of\s*)?\d{4}"
        match = re.search(pattern, query, re.IGNORECASE)
        if match:
            case = match.group(0)
            case = case.replace(" ", "_").replace("of_", "_").replace("/", "_")
            case = re.sub(r"__+", "_", case)
            return case
        return None

    # =======================================================================
    #                         HYBRID RETRIEVER
    # =======================================================================
    def _retrieve_documents(self, query: str, k: int = 10) -> List[Document]:
        if self.vectorstore is None or self.bm25 is None:
            print("⚠️ Vectorstore or BM25 not built yet.")
            return []

        # Semantic search
        semantic_results = self.vectorstore.similarity_search(query, k=k)

        # BM25 keyword search
        tokens = query.split()
        bm25_scores = self.bm25.get_scores(tokens)
        bm25_top_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:k]
        bm25_results = [self.bm25_docs[i] for i in bm25_top_idx]

        # Merge results (remove duplicates by case_number)
        combined = {}
        for doc in semantic_results + bm25_results:
            key = doc.metadata.get("case_number", doc.metadata.get("source_file", id(doc)))
            if key not in combined:
                combined[key] = doc

        return list(combined.values())[:k]

    # =======================================================================
    #                         MAIN SEARCH
    # =======================================================================
    def search(self, query: str, k: int = 10) -> List[Document]:
        print(f"\n🔍 QUERY: {query}")

        case_num = self._detect_case_number(query)
        if case_num:
            print(f"📋 Detected case number: {case_num}")
            parts = case_num.split("_")
            if len(parts) >= 2:
                case_number_only = parts[-2]
                case_year = parts[-1]

                # Get more results to filter from
                results = self._retrieve_documents(query, k=k*3)
                # Filter to exact case match (by number and year)
                exact_match = [
                    r for r in results
                    if case_number_only in r.metadata.get('case_number', '') and case_year == r.metadata.get('case_year', '')
                ]
                if exact_match:
                    results = exact_match[:k]
                    print(f"✅ Found exact case match")
                else:
                    print(f"⚠️ No exact case found, returning semantic matches")
                    results = results[:k]
            else:
                results = self._retrieve_documents(query, k=k)
        else:
            results = self._retrieve_documents(query, k=k)

        print("\n📄 Retrieved PDFs:")
        for r in results:
            print(" -", r.metadata.get("source_file", "unknown"))

        return results


In [4]:
# -------------------------------
# 1️⃣ IMPORTS
# -------------------------------
import torch
from langchain.embeddings import HuggingFaceEmbeddings

# -------------------------------
# 2️⃣ SETUP EMBEDDINGS
# -------------------------------
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

# -------------------------------
# 3️⃣ CREATE LEGAL SEARCH AGENT
# -------------------------------
agent = LegalSearchAgent(
    pdf_folder="/kaggle/input/fyp-data/supreme_court_judgments",   # path to your folder containing PDFs
    db_path="chroma_db", # folder to store vector DB
    embeddings=embeddings,
    test_mode=False       # set True to only process first 5 PDFs
)

agent.vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)

agent.build_vectordb()

# -------------------------------
# 4️⃣ RUN SEARCH
# -------------------------------
query1 = "What was CPLA 210 of 2024 about?"
results = agent.search(query1, k=15)

print("\n--- RESULTS ---")
for r in results:
    print(f"{r.metadata['case_number']} | {r.metadata['source_file']} | {r.metadata['case_year']}")


/tmp/ipykernel_48/2547412781.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

📄 Loading [750/974]: Crl.P.L.A.887-L_2013.pdf
📄 Loading [751/974]: C.A.377_2014.pdf
📄 Loading [752/974]: C.P.L.A.3062_2022.pdf
📄 Loading [753/974]: J.P.195_2017.pdf
📄 Loading [754/974]: C.M.A.12587_2021.pdf
📄 Loading [755/974]: C.P.L.A.546_2021.pdf
📄 Loading [756/974]: C.M.Appeal.39_2021.pdf
📄 Loading [757/974]: C.P.L.A.3300_2024.pdf
📄 Loading [758/974]: Crl.P.L.A.952_2021.pdf
📄 Loading [759/974]: Crl.P.L.A.1602_2023.pdf
📄 Loading [760/974]: Crl.P.L.A.668_2019.pdf
📄 Loading [761/974]: J.P.50_2023.pdf
📄 Loading [762/974]: J.P.252_2020.pdf
📄 Loading [763/974]: C.A.647_2018.pdf
📄 Loading [764/974]: Crl.A.379_2021.pdf
📄 Loading [765/974]: C.P.L.A.159_2021.pdf
📄 Loading [766/974]: C.P.L.A.394-P_2010.pdf
📄 Loading [767/974]: C.P.L.A.559-P_2024.pdf
📄 Loading [768/974]: C.P.L.A.1692-L_2020.pdf
📄 Loading [769/974]: Crl.P.L.A.1075-L_2020.pdf
📄 Loading [770/974]: Crl.P.L.A.69-Q_2022.pdf
📄 Loading [771/974]: C.P.L.A.3179-L_2023.pdf
📄 Loading [772/974]: C.A.17-Q_2023.pdf
📄 Loading [773/974]: S.M.C.

📄 Loading [775/974]: C.P.L.A.3644_2020.pdf
📄 Loading [776/974]: C.P.L.A.1290-L_2019.pdf
📄 Loading [777/974]: C.A.1414_2013.pdf
📄 Loading [778/974]: C.P.L.A.3984_2024.pdf
📄 Loading [779/974]: C.P.L.A.109-L_2024.pdf
📄 Loading [780/974]: Crl.P.L.A.231_2021.pdf
📄 Loading [781/974]: Crl.P.L.A.1690-L_2016.pdf
📄 Loading [782/974]: C.P.L.A.2987-L_2019.pdf
📄 Loading [783/974]: J.P.516_2018.pdf
📄 Loading [784/974]: Crl.A.91_2024.pdf
📄 Loading [785/974]: C.P.L.A.2537_2020.pdf
📄 Loading [786/974]: Crl.P.L.A.146_2025.pdf
📄 Loading [787/974]: Crl.A.238_2021.pdf
📄 Loading [788/974]: C.A.1444_2013.pdf
📄 Loading [789/974]: C.A.700_2014.pdf
📄 Loading [790/974]: C.A.1692_2021.pdf
📄 Loading [791/974]: C.P.L.A.2790_2018.pdf
📄 Loading [792/974]: Crl.P.L.A.1117_2024.pdf
📄 Loading [793/974]: C.P.L.A.4389_2023.pdf
📄 Loading [794/974]: C.P.L.A.414_2021.pdf
📄 Loading [795/974]: C.P.L.A.5666_2024.pdf
📄 Loading [796/974]: C.A.725_2008.pdf
📄 Loading [797/974]: C.A.875_2017.pdf
📄 Loading [798/974]: C.P.21_2022.pdf
📄

📄 Loading [829/974]: C.P.L.A.2743_2017.pdf
❌ Error loading C.P.L.A.2743_2017.pdf: Invalid Elementary Object starting with b'I' @22327: b'l)3123.3n)19( )98 0.IT\n/F2 12.0 Tf\n 0.0 0.0 rg\n0.9998 137(s)8( )-70(ne2(h)19(e)3'
📄 Loading [830/974]: C.P.L.A.5601_2021.pdf
📄 Loading [831/974]: C.A.364_2023.pdf
📄 Loading [832/974]: J.P.14_2020.pdf
📄 Loading [833/974]: C.P.L.A.1809_2020.pdf
📄 Loading [834/974]: Crl.A.525_2022.pdf
📄 Loading [835/974]: Crl.P.L.A.1016-L_2021.pdf
📄 Loading [836/974]: Crl.A.425_2019.pdf
📄 Loading [837/974]: Crl.A.36_2023.pdf
📄 Loading [838/974]: C.P.L.A.1842-L_2022.pdf
📄 Loading [839/974]: C.A.1518_2013.pdf
📄 Loading [840/974]: J.P.42_2017.pdf
📄 Loading [841/974]: Crl.A.199_2023.pdf
📄 Loading [842/974]: Crl.A.322_2018.pdf
📄 Loading [843/974]: C.P.L.A.1618_2024.pdf
📄 Loading [844/974]: C.A.248_2014.pdf
📄 Loading [845/974]: C.P.6_2023.pdf
📄 Loading [846/974]: C.A.138-L_2010.pdf
📄 Loading [847/974]: Crl.P.L.A.54_2023.pdf
📄 Loading [848/974]: Crl.P.L.A.952_2023.pdf
📄 Load

📄 Loading [873/974]: C.P.L.A.1010-L_2022.pdf
📄 Loading [874/974]: C.A.550-L_2009.pdf
📄 Loading [875/974]: C.A.151-P_2013.pdf
📄 Loading [876/974]: J.P.23_2023.pdf
📄 Loading [877/974]: Crl.A.314-L_2020.pdf
📄 Loading [878/974]: C.A.1683_2014.pdf
📄 Loading [879/974]: Crl.P.L.A.1124-L_2015.pdf
📄 Loading [880/974]: C.P.L.A.184_2024.pdf
📄 Loading [881/974]: C.A.1227_2016.pdf
📄 Loading [882/974]: C.P.L.A.1737-L_2020.pdf
📄 Loading [883/974]: C.A.350_2020.pdf
📄 Loading [884/974]: C.A.1394_2024.pdf
📄 Loading [885/974]: C.P.L.A.1422-L_2021.pdf
📄 Loading [886/974]: C.P.L.A.2478_2024.pdf
📄 Loading [887/974]: Crl.P.L.A.150-K_2024.pdf
📄 Loading [888/974]: Crl.P.L.A.435_2021.pdf
📄 Loading [889/974]: C.P.L.A.4177_2024.pdf
📄 Loading [890/974]: C.P.L.A.2475-L_2024.pdf
📄 Loading [891/974]: C.A.2434_2016.pdf
📄 Loading [892/974]: Crl.P.L.A.660_2024.pdf
📄 Loading [893/974]: C.A.700_2016.pdf
📄 Loading [894/974]: C.P.L.A.473-K_2023.pdf
📄 Loading [895/974]: C.P.L.A.1114-L_2022.pdf
📄 Loading [896/974]: C.P.L.A.15

📄 Loading [910/974]: Crl.A.81-L_2017.pdf
📄 Loading [911/974]: C.P.L.A.4618_2019.pdf
📄 Loading [912/974]: C.P.L.A.2414-L_2015.pdf
📄 Loading [913/974]: Crl.P.L.A.1288-L_2017.pdf
📄 Loading [914/974]: Crl.P.L.A.230_2019.pdf
📄 Loading [915/974]: J.P.611_2022.pdf
📄 Loading [916/974]: C.P.L.A.183_2024.pdf
📄 Loading [917/974]: C.M.A.3610_2022.pdf
📄 Loading [918/974]: C.R.P.255_2021.pdf
📄 Loading [919/974]: C.A.8-Q_2017.pdf
📄 Loading [920/974]: C.R.P.988_2023.pdf
📄 Loading [921/974]: J.P.644_2017.pdf
📄 Loading [922/974]: C.P.L.A.757-L_2021.pdf
📄 Loading [923/974]: C.P.L.A.3116_2022.pdf
📄 Loading [924/974]: C.A.139-P_2013.pdf
📄 Loading [925/974]: Crl.A.304_2020.pdf
📄 Loading [926/974]: Crl.P.L.A.806_2022.pdf
📄 Loading [927/974]: C.P.L.A.3127_2020.pdf
📄 Loading [928/974]: C.A.24-Q_2014.pdf
📄 Loading [929/974]: Crl.A.144-L_2020.pdf
📄 Loading [930/974]: Crl.A.92-L_2017.pdf
📄 Loading [931/974]: C.P.L.A.2400-L_2022.pdf
📄 Loading [932/974]: Crl.P.L.A.344_2018.pdf
📄 Loading [933/974]: C.P.L.A.1189_2025

In [5]:
queries = [
    "Why did the Supreme Court refuse to grant leave in CPLA No. 4424 of 2021?",
    "income tax ordinance section 151",
    "writ petition not maintainable",
    "What was the dispute in C.A.1509 of 2021?",
    "regularization notification 2011"
]

for q in queries:
    print(f"\n{'='*70}")
    results = agent.search(q, k=5)



🔍 QUERY: Why did the Supreme Court refuse to grant leave in CPLA No. 4424 of 2021?

📄 Retrieved PDFs:
 - C.P.L.A.694-P_2024.pdf
 - C.A.84-K_2023.pdf
 - C.P.L.A.1618_2024.pdf
 - C.P.L.A.4424_2021.pdf
 - C.A.1509_2021.pdf


🔍 QUERY: income tax ordinance section 151

📄 Retrieved PDFs:
 - C.P.L.A.2447-L_2022.pdf
 - C.A.1521_2018.pdf
 - C.P.L.A.283-L_2018.pdf
 - C.A.23_2017.pdf
 - C.P.L.A.1896_2022.pdf


🔍 QUERY: writ petition not maintainable

📄 Retrieved PDFs:
 - C.M.Appeal.39_2021.pdf
 - C.R.P.1077_2023.pdf
 - C.M.Appeal.47_2020.pdf
 - C.P.L.A.1057_2019.pdf
 - C.P.L.A.211-Q_2017.pdf


🔍 QUERY: What was the dispute in C.A.1509 of 2021?
📋 Detected case number: C.A.1509_2021
✅ Found exact case match

📄 Retrieved PDFs:
 - C.A.1509_2021.pdf


🔍 QUERY: regularization notification 2011

📄 Retrieved PDFs:
 - C.P.L.A.2210-L_2020.pdf
 - C.P.L.A.949_2023.pdf
 - C.A.864_2017.pdf
 - C.P.L.A.1010-L_2022.pdf
 - C.P.L.A.1114-L_2022.pdf


In [8]:
import shutil
import os

# 1. Copy vector store to output (if not already there)
if os.path.exists("chroma_db"):
    if os.path.exists("/kaggle/working/chroma_db"):
        shutil.rmtree("/kaggle/working/chroma_db")
    shutil.copytree("chroma_db", "/kaggle/working/chroma_db")
    print("✅ Copied chroma_db to output")

# 2. Zip the output folder
os.chdir("/kaggle/working")
shutil.make_archive("chroma_db_backup", "zip", ".", "chroma_db")
print("✅ Created chroma_db_backup.zip")

# Check file size
zip_size = os.path.getsize("chroma_db_backup.zip") / (1024**3)
print(f"📦 Zip file size: {zip_size:.2f} GB")

✅ Copied chroma_db to output
✅ Created chroma_db_backup.zip
📦 Zip file size: 0.19 GB


In [30]:
# ========================================
# 1️⃣ SETUP EMBEDDINGS
# ========================================
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

# ========================================
# 2️⃣ CREATE AGENT (don't build, just initialize)
# ========================================
agent = LegalSearchAgent(
    pdf_folder="/kaggle/input/fyp-data/supreme_court_judgments",
    db_path="chroma_db",
    embeddings=embeddings,
    test_mode=False
)

# ========================================
# 3️⃣ LOAD EXISTING VECTORSTORE
# ========================================
print("Loading existing vector store...")
agent.vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)
print(f"✅ Loaded {agent.vectorstore._collection.count()} chunks from vector DB")

# ========================================
# 4️⃣ REBUILD BM25 FROM EXISTING CHUNKS
# ========================================
print("Rebuilding BM25 index from vectorstore...")
vectorstore_data = agent.vectorstore.get()

docs = []
for i, content in enumerate(vectorstore_data['documents']):
    metadata = vectorstore_data['metadatas'][i]
    doc = Document(page_content=content, metadata=metadata)
    docs.append(doc)

agent.bm25_docs = docs
agent.bm25_corpus = [doc.page_content.split() for doc in docs]
agent.bm25 = BM25Okapi(agent.bm25_corpus)
print(f"✅ BM25 ready with {len(agent.bm25_docs)} documents")

# ========================================
# 5️⃣ NOW TEST SEARCH
# ========================================
query1 = "What was CPLA 210 of 2024 about?"
results = agent.search(query1, k=15)

print("\n--- RESULTS ---")
for r in results:
    print(f"{r.metadata['case_number']} | {r.metadata['source_file']} | {r.metadata['case_year']}")

Loading existing vector store...
✅ Loaded 87580 chunks from vector DB
Rebuilding BM25 index from vectorstore...
✅ BM25 ready with 87580 documents

🔍 QUERY: What was CPLA 210 of 2024 about?
📋 Detected case number: CPLA_210_2024
⚠️ No exact case found, returning semantic matches

📄 Retrieved PDFs:
 - Crl.P.L.A.80-P_2024.pdf
 - C.P.L.A.2226-L_2021.pdf
 - C.P.L.A.385-L_2021.pdf
 - C.P.L.A.279-Q_2020.pdf
 - C.P.L.A.1369-L_2022.pdf
 - C.P.4_2021.pdf
 - C.P.L.A.5438_2021.pdf
 - C.P.L.A.3644_2020.pdf
 - Crl.P.L.A.457-L_2021.pdf
 - Crl.P.L.A.1072_2021.pdf
 - Crl.P.L.A.112_2020.pdf
 - C.M.A.1609-L_2021.pdf
 - C.P.L.A.4806_2019.pdf
 - C.A.538_2022.pdf
 - C.P.L.A.81-P_2019.pdf

--- RESULTS ---
Crl.P.L.A.80-P_2024 | Crl.P.L.A.80-P_2024.pdf | 2024
C.P.L.A.2226-L_2021 | C.P.L.A.2226-L_2021.pdf | 2021
C.P.L.A.385-L_2021 | C.P.L.A.385-L_2021.pdf | 2021
C.P.L.A.279-Q_2020 | C.P.L.A.279-Q_2020.pdf | 2020
C.P.L.A.1369-L_2022 | C.P.L.A.1369-L_2022.pdf | 2022
C.P.4_2021 | C.P.4_2021.pdf | 2021
C.P.L.A.5438_

In [31]:
query1 = "What was C.P.L.A 210 of 2024 about?"
results = agent.search(query1, k=15)

print("\n--- RESULTS ---")
for r in results:
    print(f"{r.metadata['case_number']} | {r.metadata['source_file']} | {r.metadata['case_year']}")


🔍 QUERY: What was C.P.L.A 210 of 2024 about?

📄 Retrieved PDFs:
 - Crl.P.L.A.80-P_2024.pdf
 - C.P.L.A.279-Q_2020.pdf
 - C.P.L.A.385-L_2021.pdf
 - C.P.L.A.2226-L_2021.pdf
 - C.P.L.A.1369-L_2022.pdf
 - Crl.P.L.A.1072_2021.pdf
 - C.P.L.A.184_2024.pdf
 - Crl.P.L.A.457-L_2021.pdf
 - C.R.P.870_2023.pdf
 - C.P.L.A.3531_2021.pdf
 - C.P.L.A.6-L_2023.pdf

--- RESULTS ---
Crl.P.L.A.80-P_2024 | Crl.P.L.A.80-P_2024.pdf | 2024
C.P.L.A.279-Q_2020 | C.P.L.A.279-Q_2020.pdf | 2020
C.P.L.A.385-L_2021 | C.P.L.A.385-L_2021.pdf | 2021
C.P.L.A.2226-L_2021 | C.P.L.A.2226-L_2021.pdf | 2021
C.P.L.A.1369-L_2022 | C.P.L.A.1369-L_2022.pdf | 2022
Crl.P.L.A.1072_2021 | Crl.P.L.A.1072_2021.pdf | 2021
C.P.L.A.184_2024 | C.P.L.A.184_2024.pdf | 2024
Crl.P.L.A.457-L_2021 | Crl.P.L.A.457-L_2021.pdf | 2021
C.R.P.870_2023 | C.R.P.870_2023.pdf | 2023
C.P.L.A.3531_2021 | C.P.L.A.3531_2021.pdf | 2021
C.P.L.A.6-L_2023 | C.P.L.A.6-L_2023.pdf | 2023


In [32]:
results = agent._retrieve_documents("What was CPLA 210 of 2024 about?", k=30)
case_num = agent._detect_case_number("What was CPLA 210 of 2024 about?")

print(f"Detected case_num: {case_num}")
print(f"Looking for: number=210, year=2024")

# Extract parts
parts = case_num.split("_")
case_number_only = parts[-2] if len(parts) >= 2 else None
case_year = parts[-1] if len(parts) >= 1 else None

print(f"Extracted: case_number={case_number_only}, year={case_year}")

# Filter manually
for r in results[:5]:
    has_number = case_number_only in r.metadata['case_number']
    has_year = case_year == r.metadata['case_year']
    print(f"{r.metadata['case_number']} | year={r.metadata['case_year']} | matches={has_number and has_year}")

Detected case_num: CPLA_210_2024
Looking for: number=210, year=2024
Extracted: case_number=210, year=2024
Crl.P.L.A.80-P_2024 | year=2024 | matches=False
C.P.L.A.2226-L_2021 | year=2021 | matches=False
C.P.L.A.385-L_2021 | year=2021 | matches=False
C.P.L.A.279-Q_2020 | year=2020 | matches=False
C.P.L.A.1369-L_2022 | year=2022 | matches=False


In [35]:
# Test embedding quality
models = [
    "sentence-transformers/paraphrase-distilroberta-base-v2",
    "BAAI/bge-large-en-v1.5",
]

for model_name in models:
    emb = HuggingFaceEmbeddings(model_name=model_name)
    
    # Embed query and target
    q = emb.embed_query("[Case Type: C.P.L.A.210] [Year: 2024] [Case Number: C.P.L.A.210_2024]")
    doc = emb.embed_query("What was CPLA 210 of 2024 about?")
    
    # Cosine similarity
    import numpy as np
    sim = np.dot(q, doc) / (np.linalg.norm(q) * np.linalg.norm(doc))
    print(f"{model_name}: {sim:.4f}")

sentence-transformers/paraphrase-distilroberta-base-v2: 0.5284
BAAI/bge-large-en-v1.5: 0.7540


In [27]:
# Check if the case exists in your DB
from langchain.vectorstores import Chroma

vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)

# Search directly in metadata
all_data = vectorstore.get()
cpla_210_cases = [m for m in all_data['metadatas'] if 'C.P.L.A.210_2024' in m.get('case_number', '')]
print(f"Found {len(cpla_210_cases)} chunks from C.P.L.A.210_2024")

/tmp/ipykernel_48/1621021105.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


Found 12 chunks from C.P.L.A.210_2024


In [12]:
PDF_FOLDER = "/kaggle/input/fyp-data/supreme_court_judgments"

print("Starting LOCAL Legal Search Agent\n")
print("No API keys, No Ollama - runs 100% locally!\n")


agent = LegalSearchAgent(
    pdf_folder=PDF_FOLDER,
    test_mode=False
)

agent.build_vectordb()

print("\n" + "="*80)
print("LEGAL SEARCH AGENT READY (LOCAL)")
print("="*80)


Starting LOCAL Legal Search Agent

No API keys, No Ollama - runs 100% locally!



TypeError: LegalSearchAgent.__init__() missing 1 required positional argument: 'embeddings'

In [ ]:
print("\nType your query (or 'quit' to exit)")
print("Example: 'Why did the court reject SIC's appeal?'\n")

while True:
    query = input("\n💬 Your query: ").strip()
    
    if query.lower() in ['quit', 'exit', 'q']:
        print("\n👋 Goodbye!")
        break
    
    if not query:
        continue
    
    try:
        result = agent.search(query)
        
        print("\n📎 Relevant PDFs:")
        for pdf in result['relevant_pdfs']:
            print(f"   - {pdf}")
        print()
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Please try again or check your setup.\n")


Type your query (or 'quit' to exit)
Example: 'Why did the court reject SIC's appeal?'




💬 Your query:  What was CPLA 210 of 2024 about?



🔍 QUERY: What was CPLA 210 of 2024 about?

📄 Retrieved:
❌ Error: 'source_file'
Please try again or check your setup.




💬 Your query:  quit


In [25]:
agent = LegalSearchAgent(
    pdf_folder=PDF_FOLDER,
    test_mode=False
)
agent.build_vectordb()


🤖 Loading local LLM (Phi-2)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Local LLM loaded successfully.
📚 Loading Sentence-BERT embeddings...
Sentence-BERT loaded.

📚 Loading existing Chroma DB...
Loaded 21895 chunks.
Rebuilding BM25 index...


❌ Error loading Crl.A.306-L_2012.pdf: Invalid Elementary Object starting with b')' @12714: b'-9.9888371(h)17.56n:)-S953.72298346.2883S1(s)20.33(&)-9.988837( )-203.471(e)-27.'


❌ Error loading C.P.L.A.2743_2017.pdf: Invalid Elementary Object starting with b'I' @22327: b'l)3123.3n)19( )98 0.IT\n/F2 12.0 Tf\n 0.0 0.0 rg\n0.9998 137(s)8( )-70(ne2(h)19(e)3'


📌 BM25 index rebuilt.

